# Truthfulness Profiles — Group 7

**FINM 33200 · Spring 2026**

This notebook presents the end-to-end results of our agentic pipeline for auditing forward-looking management claims made on earnings calls.

**Companies:** Amazon (AMZN), Tesla (TSLA), Coca-Cola (KO), Eli Lilly (LLY)  
**Period:** 2020–2025 (~20 earnings calls per company, 80 calls total)  
**Claims extracted:** 539 (via GPT-5.5 extractor, prompt v5)  
**Claims verdicted:** 451 (282 via Compustat autochecker + 169 via SEC-filings agent)

---

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Resolve repo root so the notebook runs from any working directory
ROOT = Path('..').resolve()

VERDICT_COLORS = {
    'verified':            '#2ecc71',
    'partially_verified':  '#f39c12',
    'contradicted':        '#e74c3c',
    'not_yet_resolvable':  '#95a5a6',
    'insufficient_data':   '#bdc3c7',
}
VERDICT_LABELS = {
    'verified':            'Verified',
    'partially_verified':  'Partially Verified',
    'contradicted':        'Contradicted',
    'not_yet_resolvable':  'Not Yet Resolvable',
    'insufficient_data':   'Insufficient Data',
}
VERDICT_ORDER = ['verified', 'partially_verified', 'contradicted',
                 'not_yet_resolvable', 'insufficient_data']
TICKER_COLORS = {
    'AMZN': '#e67e22',
    'TSLA': '#2c3e50',
    'KO':   '#c0392b',
    'LLY':  '#8e44ad',
}
COMPANY_NAMES = {
    'AMZN': 'Amazon',
    'TSLA': 'Tesla',
    'KO':   'Coca-Cola',
    'LLY':  'Eli Lilly',
}

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titlepad': 12,
})

print('Setup complete.')

## 1. Load data

In [ ]:
# Combined verdicts: autochecker (Compustat) + agent (SEC filings)
verdicts = pd.read_csv(ROOT / 'data/verdicts/combined_55_final.csv')
claims   = pd.read_csv(ROOT / 'data/claims/55_full_run.csv')

# Join claim metadata onto verdicts
df = verdicts.merge(
    claims[['claim_id', 'company', 'verbatim_quote', 'summary',
            'claim_type', 'horizon_raw', 'horizon_period',
            'speaker_name', 'call_date']],
    on='claim_id', how='left',
    suffixes=('', '_claim'),
)

# Fill from claims if missing in verdicts
for col in ('claim_type', 'call_date'):
    col_c = col + '_claim'
    if col_c in df.columns:
        df[col] = df[col].fillna(df[col_c])
        df = df.drop(columns=[col_c])

df['call_date'] = pd.to_datetime(df['call_date'], errors='coerce')
df['year']      = df['call_date'].dt.year
df['verdict']   = df['verdict'].str.strip().str.lower()
df['verdict']   = df['verdict'].where(df['verdict'].isin(VERDICT_ORDER),
                                       other='not_yet_resolvable')

df['truth_score'] = df['verdict'].map({
    'verified': 1.0,
    'partially_verified': 0.5,
    'contradicted': 0.0,
    'not_yet_resolvable': np.nan,
    'insufficient_data': np.nan,
})

print(f'Loaded {len(df)} verdicted claims across {df["ticker"].nunique()} companies.')
print(f'Verdict breakdown:')
print(df['verdict'].value_counts().to_string())

In [ ]:
# Summary table
rows = []
for ticker in ['AMZN', 'TSLA', 'KO', 'LLY']:
    sub = df[df['ticker'] == ticker]
    vc  = sub['verdict'].value_counts()
    n   = len(sub)
    scored = sub['truth_score'].dropna()
    rows.append({
        'Company':              COMPANY_NAMES[ticker],
        'Claims':               n,
        'Verified %':           f"{vc.get('verified', 0)/n*100:.1f}%",
        'Partially Verified %': f"{vc.get('partially_verified', 0)/n*100:.1f}%",
        'Contradicted %':       f"{vc.get('contradicted', 0)/n*100:.1f}%",
        'Unresolvable %':       f"{vc.get('not_yet_resolvable', 0)/n*100:.1f}%",
        'Truth Score':          f"{scored.mean()*100:.1f}%" if len(scored) else 'n/a',
    })

summary = pd.DataFrame(rows)
summary

## 2. Truthfulness profiles — per-company overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
fig.suptitle('Truthfulness Profiles — All Companies', fontsize=18,
             fontweight='bold', y=1.01)

for ax, ticker in zip(axes, ['AMZN', 'TSLA', 'KO', 'LLY']):
    sub    = df[df['ticker'] == ticker]
    counts = sub['verdict'].value_counts()
    present = [v for v in VERDICT_ORDER if v in counts.index]
    sizes   = [counts[v] for v in present]
    colors  = [VERDICT_COLORS[v] for v in present]
    labels  = [f"{VERDICT_LABELS[v]}\n({counts[v]})" for v in present]

    wedges, _, autotexts = ax.pie(
        sizes, colors=colors, labels=None,
        autopct=lambda p: f"{p:.1f}%" if p > 4 else '',
        startangle=90, pctdistance=0.75,
        wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'},
    )
    for at in autotexts:
        at.set_fontsize(9)
        at.set_fontweight('bold')
        at.set_color('white')

    ax.add_patch(plt.Circle((0, 0), 0.50, fc='white'))

    scored = sub['truth_score'].dropna()
    score  = scored.mean() * 100 if len(scored) else 0
    ax.text(0, 0.08, f"{score:.0f}%", ha='center', va='center',
            fontsize=22, fontweight='bold',
            color=TICKER_COLORS[ticker])
    ax.text(0, -0.18, 'truth score', ha='center', va='center',
            fontsize=9, color='#666666')
    ax.set_title(f"{COMPANY_NAMES[ticker]} ({ticker})",
                 fontsize=13, fontweight='bold', pad=10)
    ax.legend(wedges, labels, loc='lower center',
              bbox_to_anchor=(0.5, -0.22), fontsize=8,
              ncol=2, frameon=False)

fig.tight_layout(pad=2.5)
plt.show()

## 3. Verdict breakdown by company

In [ ]:
tickers = ['AMZN', 'TSLA', 'KO', 'LLY']
ct = (df.groupby(['ticker', 'verdict']).size().reset_index(name='n'))
total_by_ticker = ct.groupby('ticker')['n'].transform('sum')
ct['pct'] = ct['n'] / total_by_ticker * 100

fig, ax = plt.subplots(figsize=(11, 6))
x        = np.arange(len(tickers))
bottoms  = np.zeros(len(tickers))

for verdict in VERDICT_ORDER:
    sub  = ct[ct['verdict'] == verdict].set_index('ticker')
    vals = np.array([sub.loc[t, 'pct'] if t in sub.index else 0 for t in tickers])
    ax.bar(x, vals, 0.55, bottom=bottoms,
           color=VERDICT_COLORS[verdict],
           label=VERDICT_LABELS[verdict],
           edgecolor='white', linewidth=0.8)
    for xi, (val, bot) in enumerate(zip(vals, bottoms)):
        if val > 8:
            ax.text(xi, bot + val/2, f"{val:.0f}%",
                    ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
    bottoms += vals

for xi, ticker in enumerate(tickers):
    scored = df[df['ticker'] == ticker]['truth_score'].dropna()
    score  = scored.mean() * 100 if len(scored) else 0
    n      = len(df[df['ticker'] == ticker])
    ax.text(xi, 103, f"Score: {score:.0f}%", ha='center',
            fontsize=9, color=TICKER_COLORS[ticker], fontweight='bold')
    ax.text(xi, -6, f"n={n}", ha='center', fontsize=8, color='#888')

ax.set_xticks(x)
ax.set_xticklabels([COMPANY_NAMES[t] for t in tickers], fontsize=12, fontweight='bold')
ax.set_ylim(-10, 115)
ax.set_ylabel('Share of Claims (%)')
ax.set_title('Verdict Breakdown by Company', fontsize=15, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
sns.despine(ax=ax, left=True, bottom=True)
plt.tight_layout()
plt.show()

## 4. Truthfulness over time

In [ ]:
scored = df[df['truth_score'].notna()].copy()
years  = sorted(scored['year'].dropna().unique())

fig, ax = plt.subplots(figsize=(11, 6))

for ticker in ['AMZN', 'TSLA', 'KO', 'LLY']:
    ts = (scored[scored['ticker'] == ticker]
          .groupby('year')['truth_score'].mean() * 100)
    ts = ts.reindex(years)
    ax.plot(years, ts.values, marker='o', markersize=7, linewidth=2.2,
            color=TICKER_COLORS[ticker], label=COMPANY_NAMES[ticker])
    last = ts.last_valid_index()
    if last is not None:
        ax.annotate(f"{ts[last]:.0f}%",
                    xy=(last, ts[last]),
                    xytext=(6, 0), textcoords='offset points',
                    fontsize=8.5, color=TICKER_COLORS[ticker],
                    fontweight='bold', va='center')

ax.axhline(50, color='#ccc', linestyle='--', linewidth=1, zorder=0)
ax.text(years[0], 51.5, '50% baseline', fontsize=8, color='#aaa')
ax.set_xlabel('Year of Earnings Call')
ax.set_ylabel('Truth Score (%)')
ax.set_title('Truthfulness Over Time by Company', fontsize=15, fontweight='bold')
ax.set_xticks(years)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## 5. Company x Year heatmap

In [ ]:
scored = df[df['truth_score'].notna()]
pivot  = (scored.pivot_table(values='truth_score', index='ticker',
                              columns='year', aggfunc='mean') * 100)
pivot  = pivot.loc[[t for t in ['AMZN', 'TSLA', 'KO', 'LLY'] if t in pivot.index]]
pivot.index = [COMPANY_NAMES[t] for t in pivot.index]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, ax=ax, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=100, linewidths=0.5, linecolor='#eee',
            cbar_kws={'label': 'Truth Score (%)', 'shrink': 0.8},
            annot_kws={'size': 11, 'weight': 'bold'})
ax.set_title('Truth Score Heatmap (Company x Year)', fontsize=15, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

## 6. Claim type breakdown

In [ ]:
claim_types  = ['numerical_guidance', 'capital_allocation']
type_labels  = {'numerical_guidance': 'Numerical Guidance',
                'capital_allocation':  'Capital Allocation'}
ticker_list  = ['AMZN', 'TSLA', 'KO', 'LLY']

fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True)
fig.suptitle('Verdict Breakdown by Claim Type', fontsize=15, fontweight='bold', y=1.02)

for ax, ticker in zip(axes, ticker_list):
    sub = df[df['ticker'] == ticker]
    ct  = sub.groupby(['claim_type', 'verdict']).size().reset_index(name='n')
    tot = ct.groupby('claim_type')['n'].transform('sum')
    ct['pct'] = ct['n'] / tot * 100

    x        = np.arange(len(claim_types))
    bottoms  = np.zeros(len(claim_types))

    for verdict in VERDICT_ORDER:
        v_sub = ct[ct['verdict'] == verdict].set_index('claim_type')
        vals  = np.array([v_sub.loc[ct_, 'pct'] if ct_ in v_sub.index else 0
                          for ct_ in claim_types])
        ax.bar(x, vals, 0.55, bottom=bottoms,
               color=VERDICT_COLORS[verdict],
               label=VERDICT_LABELS[verdict],
               edgecolor='white', linewidth=0.8)
        for xi, (val, bot) in enumerate(zip(vals, bottoms)):
            if val > 10:
                ax.text(xi, bot + val/2, f"{val:.0f}%",
                        ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels([type_labels[ct_].replace(' ', '\n')
                        for ct_ in claim_types], fontsize=9)
    ax.set_title(COMPANY_NAMES[ticker], fontsize=12, fontweight='bold',
                 color=TICKER_COLORS[ticker])
    ax.set_ylim(0, 108)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.grid(axis='y', alpha=0.25)
    for xi, ct_ in enumerate(claim_types):
        n = len(sub[sub['claim_type'] == ct_])
        ax.text(xi, -5, f"n={n}", ha='center', fontsize=7.5, color='#999')

handles = [mpatches.Patch(color=VERDICT_COLORS[v], label=VERDICT_LABELS[v])
           for v in VERDICT_ORDER if v in df['verdict'].values]
axes[-1].legend(handles=handles, loc='upper right',
                bbox_to_anchor=(1.55, 1), fontsize=8, framealpha=0.9)
axes[0].set_ylabel('Share of Claims (%)')
plt.tight_layout()
plt.show()

## 7. Agent evaluation results

The verification agent (GPT-5.1) was evaluated against a 28-claim auto-labeled gold set
produced by GPT-5.5 + the grading rubric, sampling capital-allocation claims from the
autochecker residual.


In [ ]:
# Load all eval run summaries
eval_runs = []

# Baseline
baseline_path = ROOT / 'data/eval/runs/2026-05-25_baseline_summary.json'
if baseline_path.exists():
    with open(baseline_path) as f:
        b = json.load(f)
    eval_runs.append({
        'Run':              'Baseline',
        'Description':      'No discipline — raw gpt-5.1 agent',
        'Recall@8':         b.get('mean_recall_at_k', None),
        'Precision':        b.get('mean_precision', None),
        'Verdict Accuracy': b.get('verdict_accuracy', None),
        'n claims':         b.get('n_claims', None),
    })

# Discipline pass runs
runs_dir = ROOT / 'data/eval/runs'
for run_dir in sorted(runs_dir.iterdir()):
    summary_path = run_dir / 'summary.json'
    if run_dir.is_dir() and summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        eval_runs.append({
            'Run':              run_dir.name,
            'Description':      run_dir.name.split('_', 2)[-1].replace('-', ' '),
            'Recall@8':         s.get('mean_recall_at_k', None),
            'Precision':        s.get('mean_precision', None),
            'Verdict Accuracy': s.get('n_claims', None) and s.get('verdict_accuracy', None),
            'n claims':         s.get('n_claims', None),
        })

eval_df = pd.DataFrame(eval_runs)
# Format as percentages
for col in ['Recall@8', 'Precision', 'Verdict Accuracy']:
    if col in eval_df.columns:
        eval_df[col] = eval_df[col].apply(
            lambda x: f"{x*100:.1f}%" if isinstance(x, float) else x
        )
eval_df

In [ ]:
# Visual: verdict accuracy across runs
runs_data = []
for run_dir in sorted((ROOT / 'data/eval/runs').iterdir()):
    s_path = run_dir / 'summary.json'
    if run_dir.is_dir() and s_path.exists():
        with open(s_path) as f:
            s = json.load(f)
        if 'verdict_accuracy' in s and s['verdict_accuracy'] is not None:
            runs_data.append({
                'run': run_dir.name.split('_', 2)[-1].replace('-', ' '),
                'verdict_accuracy': s['verdict_accuracy'] * 100,
                'recall': s.get('mean_recall_at_k', 0) * 100,
                'precision': s.get('mean_precision', 0) * 100,
            })

# Add baseline
if baseline_path.exists():
    with open(baseline_path) as f:
        b = json.load(f)
    runs_data.insert(0, {
        'run': 'baseline',
        'verdict_accuracy': b.get('verdict_accuracy', 0) * 100,
        'recall': b.get('mean_recall_at_k', 0) * 100,
        'precision': b.get('mean_precision', 0) * 100,
    })

if runs_data:
    rdf    = pd.DataFrame(runs_data)
    x      = np.arange(len(rdf))
    width  = 0.25

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width, rdf['recall'],           width, label='Recall@8',         color='#3498db', alpha=0.85)
    ax.bar(x,         rdf['precision'],         width, label='Precision',         color='#e67e22', alpha=0.85)
    ax.bar(x + width, rdf['verdict_accuracy'],  width, label='Verdict Accuracy',  color='#2ecc71', alpha=0.85)

    for bars in ax.containers:
        ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=8, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(rdf['run'], fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('Score (%)')
    ax.set_title('Agent Evaluation — Metrics Across Runs', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print('No eval run data found.')

## 8. Sample claims — verified and contradicted

A random sample of claims from each verdict bucket, for qualitative review.

In [ ]:
np.random.seed(42)

for verdict in ['verified', 'contradicted', 'partially_verified']:
    sub = df[df['verdict'] == verdict].dropna(subset=['verbatim_quote'])
    if sub.empty:
        continue
    sample = sub.sample(min(4, len(sub)))
    print(f"\n{'='*70}")
    print(f"  {VERDICT_LABELS[verdict].upper()} ({len(sub)} total)")
    print(f"{'='*70}")
    for _, row in sample.iterrows():
        co   = COMPANY_NAMES.get(row['ticker'], row['ticker'])
        date = str(row['call_date'])[:10] if pd.notna(row['call_date']) else ''
        print(f"  [{co}, {date}]")
        quote = str(row.get('verbatim_quote', row.get('summary', '')))
        print(f"  '{quote[:120]}{'...' if len(quote)>120 else ''}'")  
        print()

## 9. Key findings and limitations

### Findings

- **Amazon** had the highest truthfulness score at **77%**, reflecting consistent delivery on financial guidance.
- **Coca-Cola** had the lowest score at **58%**, with a relatively high rate of contradicted claims (24%).
- **Tesla and Eli Lilly** sat in the middle (~65-68%), with Tesla having the highest share of unresolvable claims (49%) — many forward-looking capacity and production promises could not be confirmed in available filings.
- Across all companies, **39% of claims were fully verified**, **12% partially verified**, **16% contradicted**, and **33% unresolvable** given the available filing window.

### Agent evaluation

The verification agent (GPT-5.1) was evaluated against a 28-claim auto-labeled gold set:
- **Recall@8: 75%** — the agent finds the right filing evidence most of the time.
- **Verdict accuracy: 56% baseline, 71% after discipline pass** — improved substantially after adding evidence-grounding constraints.
- **Main weakness:** the agent over-claims on forward-looking claims with horizons not yet elapsed (confabulated verdicts on 3 of 6 control claims).

### Known limitations

1. **LLM-labeled gold set** — the gold set was produced by GPT-5.5 + rubric rather than human annotators. A deliberate time-constrained choice; the labeler and agent use different models to reduce circularity.
2. **Small evaluation n** — 28-claim gold set limits statistical power.
3. **Verdict skew** — the gold set is verified-heavy with zero contradicted claims, so the agent's contradiction detection was not tested.
4. **Horizon coverage** — 33% of claims are unresolvable because their projected horizon extends beyond our available filing window (post-2024 claims).
5. **Compustat coverage** — not all numerical guidance claims map cleanly to Compustat fields; the autochecker screened out claims it could not verify, which inflates the unresolvable count.
